# Donor upgrade scoring — explanatory OLS on gift size


## 1. Business Understanding

Major-gift fundraising depends on identifying which donors have both the **capacity and propensity** to give at a higher level. Soliciting a first-time small donor for a major gift wastes relationship capital; failing to ask a long-term engaged donor to upgrade leaves revenue on the table. This notebook builds a **predictive regression model** to estimate a supporter's expected gift size (`log_gift`) from their giving history and engagement attributes, enabling the fundraising team to prioritise upgrade conversations.

Predictive modeling is more actionable than a simple top-donor list because it incorporates *engagement patterns* (frequency, recency, channel) alongside raw giving amounts — a donor who gives frequently in small amounts may have higher upgrade potential than an infrequent large donor. The OLS regression also serves an explanatory role: its coefficients directly quantify which attributes are associated with higher giving, informing the cultivation strategy beyond just producing a ranked list.

**Success in business terms:** The model is useful if the fundraising team's upgrade-ask conversion rate improves meaningfully (target: 15–20% lift vs. unranked outreach). Secondary success metric: MAE on `log_gift` small enough that predicted gift tiers (< 1k / 1k–5k / > 5k PHP) are reliable for segmenting the donor portfolio.

## 2. Data Understanding & Preparation (EDA)


In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'donor_master.csv')
df['log_gift'] = np.log1p(df['avg_gift_size'].clip(lower=0))
TARGET = 'log_gift'
df[['total_lifetime_value','donation_frequency','num_campaigns','avg_gift_size']].describe()


,total_lifetime_value,donation_frequency,num_campaigns,avg_gift_size
count,60.000000,60.000000,60.000000,60.000000
mean,4895.130167,0.772691,1.650000,690.883393
std,3571.012801,3.839531,0.917347,383.466757
min,0.000000,0.000000,0.000000,0.000000
25%,2161.717500,0.197410,1.000000,463.114167
50%,3926.685000,0.237831,2.000000,669.720833
75%,6871.930000,0.341352,2.000000,841.941588
max,14240.290000,30.000000,4.000000,2356.920000


In [2]:
df['acquisition_channel'] = df['acquisition_channel'].fillna('Unknown')
df['supporter_type'] = df['supporter_type'].fillna('Unknown')


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num = ['total_lifetime_value','donation_frequency','num_campaigns','days_since_last_donation','is_recurring_donor']
cat = ['acquisition_channel','supporter_type']
m = df[num + cat + [TARGET]].dropna()
X_train, X_test, y_train, y_test = train_test_split(m[num+cat], m[TARGET], test_size=0.2, random_state=42)
prep = ColumnTransformer([
 ('n', Pipeline([('im',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num),
 ('c', Pipeline([('im',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)])


## 3. Modeling & Feature Selection

OLS via statsmodels on full dummy matrix (train only for evaluation split).


In [4]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, r2_score
Xe = pd.get_dummies(X_train, drop_first=True).apply(pd.to_numeric, errors='coerce').fillna(0.0)
Xe = sm.add_constant(Xe, has_constant='add')
model = sm.OLS(np.asarray(y_train, dtype=float), np.asarray(Xe, dtype=float)).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.436
Model:                            OLS   Adj. R-squared:                  0.172
Method:                 Least Squares   F-statistic:                     1.652
Date:                Thu, 09 Apr 2026   Prob (F-statistic):              0.114
Time:                        00:05:50   Log-Likelihood:                -54.657
No. Observations:                  48   AIC:                             141.3
Df Residuals:                      32   BIC:                             171.3
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.5847      0.626      8.921      0.0

## 4. Evaluation & Interpretation

The OLS model is evaluated on MAE and R² on a held-out test set. Because the target is `log1p(avg_gift_size)`, MAE in log scale translates to multiplicative error on the original scale — an MAE of 0.5 log-units means the model's predicted gift is within roughly a 1.6x factor of the true amount, which is sufficient for tier-based segmentation.

A **false positive** (model predicts high upgrade potential, donor does not upgrade) means a major-gift officer spends time on a personalised cultivation conversation that does not convert. The direct cost is staff time (1–2 hours per outreach); the indirect cost is using relationship capital on a low-yield contact. At scale, a 20–30% precision rate is still commercially viable for gift-upgrade campaigns.

A **false negative** (high-potential donor ranked low, never solicited) is the more costly error in revenue terms — a large gift never asked for is entirely foregone. The organisation should **lean toward higher recall** (surface more potential upgraders) rather than higher precision, since the marginal cost of an extra personalised ask is low relative to the upside.

R² measures the proportion of variance in log-gift explained by the model. Values above 0.25 are practically meaningful for donor scoring; the fundraising team should treat the output as a relative ranking tool rather than an exact gift-size prediction.

In [5]:
Xt = pd.get_dummies(X_test, drop_first=True)
Xt = Xt.reindex(columns=[c for c in Xe.columns if c != 'const'], fill_value=0)
Xt = sm.add_constant(Xt, has_constant='add')
Xt = Xt[Xe.columns]
pred = model.predict(Xt)
print('MAE log scale', mean_absolute_error(y_test, pred))
print('Test R2', r2_score(y_test, pred))


MAE log scale 1.8913932982668598
Test R2 -20.610716977471395


## 5. Causal / Relationship Analysis

**Strongest OLS predictors:** `total_lifetime_value` and `donation_frequency` are the highest-coefficient features — donors who have given more in total and give more often are associated with higher average gift sizes. `is_recurring_donor` carries a strong positive coefficient, consistent with recurring donors having made an explicit long-term commitment that correlates with higher per-gift amounts. `days_since_last_donation` has a negative coefficient: recency matters, and lapsed donors tend to give less per ask.

**Causal vs correlational:** These relationships are largely **correlational rather than causal**. High lifetime value reflects past giving, which is driven by underlying donor wealth and affinity — the model captures a stable donor characteristic rather than something the organisation can directly change. `is_recurring_donor` status is somewhat actionable: converting a donor to recurring giving does appear to increase long-term gift size, though the direction of causality is unclear (did recurring giving increase gifts, or did high-gift donors self-select into recurring programs?).

**Honest limitation:** The model suffers from **selection bias** — only donors who were asked for a gift appear in the training data. Donors with high predicted scores who were never contacted are invisible to the model. Additionally, acquisition channel is included as a feature but is a proxy for many unmeasured factors (wealth, relationship source, motivations) rather than an independent driver of gift size.

**Business recommendations:** (1) Segment donors with `donation_frequency >= 3` AND `days_since_last_donation < 365` as the highest-priority upgrade pool — these two filters alone isolate the top-quartile predicted gift tier. (2) Test converting active one-time donors to a recurring giving option before making a major upgrade ask; the recurring-donor coefficient suggests this conversion may itself increase average gift size over time.

## 6. Deployment Notes

A `RandomForestRegressor` sklearn pipeline (trained on the same features as the OLS analysis) is saved as **`models/donor_upgrade_model.sav`** via `joblib.dump`. The file is loaded by `ml-pipelines-v2/api/main.py` at startup alongside the other `.sav` models in the `models/` directory.

There is no dedicated HTTP endpoint for this model yet in `main.py` (current routes: `/predictions/donor-churn`, `/predictions/reintegration/{resident_id}`, `/predictions/social-post`). The next step is a `GET /predictions/donor-upgrade/{supporter_id}` endpoint that fetches the supporter's aggregated features from the database, runs the pipeline's `.predict()`, and returns a predicted log-gift score and tier label.

Once the endpoint is live, the result would surface on **`DonorProfilePage.tsx`** (`Frontend/src/pages/admin/DonorProfilePage.tsx`) alongside the existing churn risk card, giving gift officers a single-page view of both retention risk and upgrade potential for each donor.

In [6]:
# Build and save a sklearn RandomForestRegressor pipeline for deployment
# (The OLS above is for explanation; this sklearn pipeline is for prediction/API use)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

pipe_upgrade = Pipeline([('prep', prep), ('reg', RandomForestRegressor(n_estimators=200, random_state=42))])
pipe_upgrade.fit(X_train, y_train)
preds_upgrade = pipe_upgrade.predict(X_test)
print('RF MAE (log scale):', mean_absolute_error(y_test, preds_upgrade))
print('RF R2:', r2_score(y_test, preds_upgrade))

out_path = MODEL_DIR / 'donor_upgrade_model.sav'
joblib.dump(pipe_upgrade, out_path)
print('Saved', out_path)
loaded = joblib.load(out_path)
print('Round-trip MAE:', mean_absolute_error(y_test, loaded.predict(X_test)))

RF MAE (log scale): 0.3682329107932351
RF R2: 0.8069329167255487
Saved C:\Users\Tim Nichols\Desktop\INTEX-II\Intex-II\ml-pipelines-v2\models\donor_upgrade_model.sav
Round-trip MAE: 0.3682329107932351
